In [1]:
# Tune Elastic Net logistic regression on training and validation data.
# Freeze the validation threshold, evaluate the test once, and save the pipeline and report.


In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, file_hash, probability_metrics, calibration_table, embedded_image, load_split
from pathlib import Path
import base64
import hashlib
import html
import inspect
import io
import json
import platform
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.calibration import calibration_curve
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, auc, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_recall_curve,
    roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data/processed/train_matrix.csv').exists(): ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
REPORTS = ROOT / 'artifacts/19_model_elastic_net_logistic'
MODELS = project_paths(ROOT).models
FIGURES = REPORTS
C_GRID = [.001, .01, .1, 1., 10.]
L1_RATIOS = [.1, .5, .9]
CLASS_WEIGHTS = {'unweighted': None, 'balanced': 'balanced'}
MAX_ITER, TOL, SEED = 12000, 1e-4, 42
input_hashes = {}


artifact_path = MODELS / 'preprocessing_pipeline.joblib'
input_hashes[artifact_path] = file_hash(artifact_path)
preprocessing = joblib.load(artifact_path)
schema = preprocessing.project_schema_
FEATURES, ID_COLUMNS = schema['matrix_feature_columns'], schema['matrix_metadata_columns']
preprocessing_state = joblib.hash(preprocessing)


train, validation = load_split('train', PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes), load_split('validation', PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
train_before, validation_before = train.copy(deep=True), validation.copy(deep=True)
selector = VarianceThreshold(threshold=0.).fit(train[FEATURES].to_numpy(dtype=float))
keep = selector.get_support()
ACTIVE_FEATURES = np.asarray(FEATURES)[keep].tolist()
EXCLUDED_CONSTANTS = np.asarray(FEATURES)[~keep].tolist()
X_train = sparse.csr_matrix(selector.transform(train[FEATURES].to_numpy(dtype=float)))
X_validation = sparse.csr_matrix(selector.transform(validation[FEATURES].to_numpy(dtype=float)))
y_train, y_validation = train.target.to_numpy(dtype=int), validation.target.to_numpy(dtype=int)
selector_state = joblib.hash(selector)


In [3]:
# Define metrics once, using target=1 for bankruptcy and missing-safe confusion counts.

# sklearn >=1.8 specifies Elastic Net through l1_ratio; earlier releases need penalty.
def make_classifier(C, l1_ratio, class_weight):
    parameters = dict(C=C, l1_ratio=l1_ratio, class_weight=class_weight, solver='saga',
                      max_iter=MAX_ITER, tol=TOL, random_state=SEED, warm_start=False)
    if inspect.signature(LogisticRegression).parameters['penalty'].default != 'deprecated':
        parameters['penalty'] = 'elasticnet'
    return LogisticRegression(**parameters)

# Every fit sees only training data. Convergence is part of candidate eligibility.
candidates, tuning_rows = {}, []
for weight_name, class_weight in CLASS_WEIGHTS.items():
    for l1_ratio in L1_RATIOS:
        for C in C_GRID:
            candidate_id = f'{weight_name}_l1_{l1_ratio:g}_C_{C:g}'
            model = make_classifier(C, l1_ratio, class_weight)
            started = time.perf_counter()
            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter('always')
                model.fit(X_train, y_train)
            convergence_warning = any(issubclass(w.category, ConvergenceWarning) for w in captured)
            finite_model = np.isfinite(model.coef_).all() and np.isfinite(model.intercept_).all()
            eligible = bool(not captured and not convergence_warning and model.n_iter_.max() < MAX_ITER and finite_model)
            row = {'candidate_id': candidate_id, 'C': C, 'l1_ratio': l1_ratio, 'class_weight': weight_name,
                   'eligible': eligible, 'iterations': int(model.n_iter_.max()),
                   'nonzero_coefficients': int(np.count_nonzero(model.coef_)),
                   'elapsed_seconds': time.perf_counter() - started,
                   'warnings': '; '.join(sorted({str(w.message) for w in captured}))}
            metrics = probability_metrics(y_validation, model.predict_proba(X_validation)[:, 1])
            row.update({'validation_' + k: v for k, v in metrics.items()})
            tuning_rows.append(row)
            if eligible: candidates[candidate_id] = model
tuning = pd.DataFrame(tuning_rows)
eligible = tuning.loc[tuning.eligible].sort_values(
    ['validation_pr_auc_average_precision', 'validation_roc_auc', 'validation_brier_score',
     'nonzero_coefficients', 'C', 'l1_ratio', 'class_weight'],
    ascending=[False, False, True, True, True, False, True], kind='stable')
if eligible.empty: raise RuntimeError('No candidate converged; inspect solver warnings before test evaluation')
best_row = eligible.iloc[0]
best_model = candidates[best_row.candidate_id]
validation_probability = best_model.predict_proba(X_validation)[:, 1]
tuning['selected'] = tuning.candidate_id.eq(best_row.candidate_id)


In [4]:
# Enumerate all validation classification partitions, including all-positive/negative.
threshold_candidates = np.unique(np.r_[0., .5, validation_probability, np.nextafter(validation_probability.max(), np.inf)])
threshold_rows = pd.DataFrame([probability_metrics(y_validation, validation_probability, value) for value in threshold_candidates])
threshold_rows['distance_from_half'] = (threshold_rows.threshold - .5).abs()
threshold_rows = threshold_rows.sort_values(
    ['balanced_accuracy', 'sensitivity', 'specificity', 'distance_from_half', 'threshold'],
    ascending=[False, False, False, True, False], kind='stable')
OPERATING_THRESHOLD = float(threshold_rows.iloc[0].threshold)

# Freeze both decisions and the trained model before reading the held-out test data.
model_configuration = {'selected_candidate': best_row.candidate_id, 'C': float(best_row.C),
    'l1_ratio': float(best_row.l1_ratio), 'class_weight': best_row.class_weight,
    'operating_threshold': OPERATING_THRESHOLD, 'fit_split': 'train',
    'selection_split': 'validation', 'selection_metric': 'average_precision',
    'threshold_metric': 'balanced_accuracy', 'solver': 'saga', 'max_iter': MAX_ITER,
    'tol': TOL, 'seed': SEED, 'iterations': int(best_model.n_iter_.max()),
    'active_predictors': len(ACTIVE_FEATURES), 'training_constant_predictors': EXCLUDED_CONSTANTS,
    'preprocessing_sha256': input_hashes[artifact_path],
    'training_event_ids': train.metadata_event_id.tolist(),
    'validation_event_ids': validation.metadata_event_id.tolist(),
    'predict_default_threshold': .5, 'calibration': 'No post-hoc calibrator; sampled-cohort probabilities'}
configuration_state = joblib.hash(model_configuration)
model_state = joblib.hash(best_model)

# A single test evaluation follows, without fitting or changing any decision.
test = load_split('test', PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
test_before = test.copy(deep=True)
X_test = sparse.csr_matrix(selector.transform(test[FEATURES].to_numpy(dtype=float)))
y_test = test.target.to_numpy(dtype=int)
probabilities = {'train': best_model.predict_proba(X_train)[:, 1],
                 'validation': validation_probability, 'test': best_model.predict_proba(X_test)[:, 1]}
frames = {'train': train, 'validation': validation, 'test': test}
metric_rows, prediction_frames = [], []
for split, frame in frames.items():
    for rule, threshold in [('default_0.5', .5), ('validation_selected', OPERATING_THRESHOLD)]:
        metric_rows.append({'split': split, 'rule': rule, **probability_metrics(frame.target.to_numpy(), probabilities[split], threshold)})
    prediction_frame = frame[ID_COLUMNS].copy()
    prediction_frame['bankruptcy_probability'] = probabilities[split]
    prediction_frame['predicted_target_default_0_5'] = (probabilities[split] >= .5).astype(int)
    prediction_frame['predicted_target_operating'] = (probabilities[split] >= OPERATING_THRESHOLD).astype(int)
    prediction_frames.append(prediction_frame)
metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)


In [5]:
# Map penalized coefficients back to every notebook 17 matrix column.
all_coefficients = np.zeros(len(FEATURES))
all_coefficients[keep] = best_model.coef_[0]
coefficient_rows = []
for feature, coefficient, active in zip(FEATURES, all_coefficients, keep):
    branch, source = feature.split('__', 1)
    if branch == 'numeric':
        family = source.split('_', 1)[0]
        unit = 'one training SD of median-imputed value'
    elif branch == 'missing':
        source = source.removeprefix('missingindicator_')
        family = 'missing_' + source.split('_', 1)[0]
        unit = 'missing indicator: 0 to 1'
    else:
        family = 'categorical_' + ('sector' if source.startswith('sector_') else 'industry')
        unit = 'category indicator: 0 to 1 (other predictors held fixed)'
    coefficient_rows.append({'feature': feature, 'family': family, 'effect_unit': unit,
        'included_in_model': bool(active), 'excluded_reason': '' if active else 'constant in training',
        'coefficient': float(coefficient), 'absolute_coefficient': abs(float(coefficient)),
        'odds_ratio': float(np.exp(coefficient)), 'nonzero': bool(active and coefficient != 0)})
coefficients = pd.DataFrame(coefficient_rows).sort_values(['absolute_coefficient', 'feature'], ascending=[False, True], kind='stable')
family_counts = coefficients.groupby('family', sort=True).agg(
    candidate_features=('feature', 'size'), active_features=('included_in_model', 'sum'),
    nonzero_coefficients=('nonzero', 'sum'), absolute_coefficient_sum=('absolute_coefficient', 'sum')).reset_index()
final_pipeline = Pipeline([('preprocessing', preprocessing), ('training_constant_selector', selector), ('classifier', best_model)])
final_pipeline.project_model_ = dict(model_configuration)
final_pipeline.project_model_['matrix_feature_columns'] = FEATURES
final_pipeline.project_model_['active_feature_columns'] = ACTIVE_FEATURES
final_pipeline.project_model_['intercept'] = float(best_model.intercept_[0])
final_pipeline.project_model_['versions'] = {'python': platform.python_version(), 'sklearn': sklearn.__version__,
    'numpy': np.__version__, 'pandas': pd.__version__, 'joblib': joblib.__version__}


In [6]:
# Plot reliability against outcomes, without fitting a calibration transformation.

for directory in [REPORTS, MODELS]: directory.mkdir(parents=True, exist_ok=True)
calibration_frames = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5.3), constrained_layout=True)
colors = {'train': '#777777', 'validation': '#d98300', 'test': '#1769aa'}
for split in ['train', 'validation', 'test']:
    labels, probability = frames[split].target.to_numpy(), probabilities[split]
    table = calibration_table(labels, probability)
    table['split'] = split
    calibration_frames.append(table)
    axes[0].plot(table.mean_predicted_probability, table.observed_bankruptcy_rate,
        marker='o', color=colors[split], label=f'{split} (n={len(labels)})', alpha=.8)
    if split == 'test':
        for row in table.itertuples(): axes[0].annotate(f'n={row.n}', (row.mean_predicted_probability, row.observed_bankruptcy_rate),
            xytext=(3, 5), textcoords='offset points', fontsize=8)
    axes[1].hist(probability, bins=np.linspace(0, 1, 16), histtype='step', linewidth=1.8,
                 color=colors[split], label=split, density=True)
axes[0].plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1, label='perfect calibration')
axes[0].set(xlabel='Mean predicted bankruptcy probability', ylabel='Observed bankruptcy fraction',
            title='Reliability: quantile bins (up to eight)', xlim=(-.02, 1.02), ylim=(-.02, 1.02))
axes[0].legend(fontsize=8); axes[0].grid(alpha=.2)
axes[1].axvline(OPERATING_THRESHOLD, color='black', linestyle='--', label=f'operating threshold {OPERATING_THRESHOLD:.3f}')
axes[1].set(xlabel='Predicted bankruptcy probability', ylabel='Density', title='Probability distributions by chronological split')
axes[1].legend(fontsize=8); axes[1].grid(alpha=.2)
fig.suptitle('Elastic Net logistic regression: sampled-cohort calibration')
calibration_path = FIGURES / 'elastic_net_calibration.png'
fig.savefig(calibration_path, dpi=180, bbox_inches='tight'); plt.close(fig)
calibration_bins = pd.concat(calibration_frames, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.3), constrained_layout=True)
for split in ['validation', 'test']:
    labels, probability = frames[split].target.to_numpy(), probabilities[split]
    fpr, tpr, _ = roc_curve(labels, probability)
    precision, recall, _ = precision_recall_curve(labels, probability)
    axes[0].plot(fpr, tpr, color=colors[split], label=f'{split}: AUC={roc_auc_score(labels, probability):.3f}')
    axes[1].plot(recall, precision, color=colors[split], label=f'{split}: AP={average_precision_score(labels, probability):.3f}')
    axes[1].axhline(labels.mean(), color=colors[split], linestyle=':', alpha=.6, label=f'{split} prevalence={labels.mean():.3f}')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
axes[0].set(xlabel='False positive rate', ylabel='Sensitivity', title='ROC curves', xlim=(0, 1), ylim=(0, 1.02))
axes[1].set(xlabel='Recall / sensitivity', ylabel='Precision', title='Precision-recall curves', xlim=(0, 1), ylim=(0, 1.02))
for ax in axes: ax.legend(fontsize=9); ax.grid(alpha=.2)
fig.suptitle('Chronological validation selection and held-out test performance')
roc_pr_path = FIGURES / 'elastic_net_roc_pr.png'
fig.savefig(roc_pr_path, dpi=180, bbox_inches='tight'); plt.close(fig)


In [7]:
# Export a reviewable benchmark report with parameters, metrics, and interpretation.
def table_html(frame):
    return '<div class="table-scroll">' + frame.to_html(index=False, escape=True,
        float_format=lambda value: f'{value:.5g}', na_rep='not available', border=0) + '</div>'


split_summary = pd.DataFrame([{'split': split, 'events': len(frame), 'companies': frame.company_id.nunique(),
    'company_groups': frame.company_group_id.nunique(), 'bankrupt_events': int(frame.target.sum()),
    'prevalence': frame.target.mean(), 'first_prediction': frame.prediction_date.min(),
    'last_prediction': frame.prediction_date.max(), 'largest_absolute_matrix_value': abs(frame[FEATURES].to_numpy()).max()}
    for split, frame in frames.items()])
provenance = pd.DataFrame([{'input': str(path.relative_to(ROOT)), 'sha256': digest} for path, digest in input_hashes.items()])
search_boundary = best_row.C in [min(C_GRID), max(C_GRID)]
selection_summary = pd.DataFrame([{'parameter': key, 'value': str(model_configuration[key])} for key in
    ['selected_candidate', 'C', 'l1_ratio', 'class_weight', 'operating_threshold', 'iterations', 'active_predictors']])
report_sections = [
    '<h1>Elastic Net logistic regression benchmark</h1>',
    '<p>Primary interpretable model. All candidate classifiers and preprocessing fit training observations only. '
    'Hyperparameters and the operating threshold are selected on chronological validation; the later test is evaluated '
    'after those decisions are frozen. No random split, correlated-feature filter, post-hoc calibration, or held-out refit is used.</p>',
    '<h2>Chronological cohorts and input scale</h2>', table_html(split_summary),
    '<p>Training constants are excluded using a training-fitted VarianceThreshold; all other predictors, including correlated '
    'groups and aliases, are retained. Matrix values are neither clipped nor winsorized. Large held-out standardized values '
    'reflect the frozen training scales and may create very confident probabilities.</p>',
    '<h2>Selected hyperparameters and operating rule</h2>', table_html(selection_summary),
    f'<p>Grid: C={C_GRID}; l1_ratio={L1_RATIOS}; class weights={list(CLASS_WEIGHTS)}. '
    f'{int(tuning.eligible.sum())} of {len(tuning)} candidates were eligible after solver checks. '
    f'Selected C lies on the grid boundary: {search_boundary}. Smaller C means stronger regularization. '
    'Selection ranks validation average precision, then ROC-AUC, lower Brier, fewer coefficients, smaller C, higher L1 mix, '
    'and class-weight name. All trial metrics remain available in the tuning CSV.</p>',
    '<p>The operating threshold maximizes validation balanced accuracy, with sensitivity, specificity, closeness to 0.5, '
    'and higher threshold used as tie-breakers. Every exported probability uses the same training-fitted model. '
    'The classifier is not refitted on training plus validation. Pipeline.predict uses 0.5; apply the stored operating '
    'threshold to predict_proba for the selected rule.</p>',
    '<h2>Performance</h2>', table_html(metrics),
    '<p>PR-AUC primary summary is average precision (AP), a recall-weighted precision sum. Trapezoidal PR area is separately '
    'reported. Sensitivity is bankruptcy recall; specificity is non-bankruptcy recall. Brier is the unweighted mean '
    'squared probability error on each split. Training results are in-sample diagnostics; validation results reflect '
    'model/threshold selection; test results are held-out evaluation. No confidence interval is inferred from these point estimates.</p>',
    embedded_image(roc_pr_path, 'Validation and held-out test ROC and precision-recall curves.'),
    '<h2>Calibration</h2>', embedded_image(calibration_path, 'Uncalibrated model reliability with test-bin sample counts.'),
    table_html(calibration_bins),
    '<p>Reliability curves are empirical diagnostics, not a fitted calibration transformation. Quantile ties can leave fewer '
    'than eight bins. Small bins are noisy. Class weights and matched sampling alter prevalence, so probabilities and Brier '
    'scores describe this sampled cohort rather than population default risk.</p>',
    '<h2>Penalized coefficients and interpretation</h2>', table_html(family_counts),
    f'<p>Intercept: {float(best_model.intercept_[0]):.6g}; nonzero coefficients: {int(coefficients.nonzero.sum())}. '
    'Odds ratios are exp(coefficient) conditional on other included predictors. Numeric effects represent one training '
    'standard deviation of an imputed value; binary indicators represent 0 to 1. These are penalized associations, with '
    'no causal interpretation or unpenalized p-values. Correlated group members can share effects and have unstable '
    'individual coefficients. Categorical dummy effects do not define a unique reference-category comparison.</p>',
    table_html(coefficients.loc[coefficients.nonzero].head(50)),
    '<p>Full coefficients, including regularized zeros and excluded constants, are in elastic_net_coefficients.csv.</p>',
    '<h2>Validation search audit</h2>', table_html(tuning.sort_values(['eligible', 'validation_pr_auc_average_precision'], ascending=[False, False])),
    '<details><summary>Validation threshold candidates (top ten)</summary>' + table_html(threshold_rows.head(10)) + '</details>',
    '<h2>Scope and reproducibility</h2>',
    '<p>The fitted artifact contains the frozen raw-data preprocessing, constant selector, and classifier using built-in '
    'sklearn components. project_model_ stores the selected operating threshold, training/validation IDs, feature names, '
    'configuration, hashes, and software versions. Inputs are checked unchanged after evaluation and serialization.</p>',
    '<p>Upstream limitations remain: notebook 14 flagged the requested 1:2 design versus the existing 1:3 matched source; '
    'time grouping/purging can break individual matched sets. Independent negative-outcome surveillance, historical '
    'entity/classification continuity, and macro release/vintage dates are not fully verified by the source tables. '
    'This benchmark does not resolve those earlier evidence gaps.</p>', table_html(provenance),
    '<p>Software: ' + html.escape(json.dumps(final_pipeline.project_model_['versions'], sort_keys=True)) + '</p>'
]
report_html = ('<!doctype html><html lang="en"><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width,initial-scale=1"><title>Elastic Net logistic benchmark</title>'
    '<style>body{font:15px/1.55 system-ui,sans-serif;margin:32px auto;padding:0 24px;max-width:1280px;color:#172b3a}'
    'h1,h2{color:#123b59}p{max-width:1100px}.table-scroll{overflow:auto;max-height:560px;margin:16px 0}'
    'table{border-collapse:collapse;font-size:12px;width:100%}th,td{padding:7px;text-align:left;border-bottom:1px solid #d8e1e8;white-space:nowrap}'
    'th{position:sticky;top:0;background:#eaf1f6}tr:nth-child(even){background:#f7f9fb}'
    'figure{margin:24px 0}img{max-width:100%;height:auto}figcaption{font-size:13px;color:#52636e}'
    'details{margin:16px 0}summary{cursor:pointer}</style></head><body>' + '\n'.join(report_sections) + '</body></html>')


In [8]:
# Verify keys, no refits, metric arithmetic, and raw-input inference before publishing.


# Raw rows are used only to check frozen inference, after every decision was locked.
raw_path = PROCESSED / 'modeling_dataset_unprocessed.csv'
input_hashes[raw_path] = file_hash(raw_path)
raw = pd.read_csv(raw_path, usecols=['company_id', 'prediction_date'] + schema['input_feature_columns'], dtype={'company_id': 'string'})
raw = raw.set_index(['company_id', 'prediction_date'])
for split, frame in frames.items():
    rows = raw.reindex(pd.MultiIndex.from_frame(frame[['company_id', 'prediction_date']]))

final_pipeline.project_model_['input_sha256'] = {str(path.relative_to(ROOT)): digest for path, digest in input_hashes.items()}
final_pipeline.project_model_['test_event_ids'] = test.metadata_event_id.tolist()

model_path = MODELS / 'elastic_net_logistic.joblib'
joblib.dump(final_pipeline, model_path)
loaded = joblib.load(model_path)
probe_frame = train.iloc[:12]
probe_rows = raw.reindex(pd.MultiIndex.from_frame(probe_frame[['company_id', 'prediction_date']]))
tuning.to_csv(REPORTS / 'elastic_net_tuning_results.csv', index=False)
metrics.to_csv(REPORTS / 'elastic_net_metrics.csv', index=False)
coefficients.to_csv(REPORTS / 'elastic_net_coefficients.csv', index=False)
predictions.to_csv(REPORTS / 'elastic_net_predictions.csv', index=False)
(REPORTS / 'elastic_net_logistic_report.html').write_text(report_html, encoding='utf-8')


417266